# TARO Phase 3 All in One Notebook

This is the only notebook needed for the project. It contains an embedded copy of the TARO source code and tests. When opened from Windows, it creates a separate Ubuntu run folder, installs the environment, runs the verified simulator tests, executes the targeted burst experiment, and displays the evidence graphs.

The targeted experiment is documented as an exploratory scenario. It demonstrates a measured TARO stability and delivery tradeoff; it does not claim that a local heuristic reproduces or universally beats Grace, GTSR, ERA-MADDPG, CLPE-TE or other published methods.

## What this notebook does

1. Creates taro_one_notebook_run.
2. Extracts the embedded TARO source and tests into that folder.
3. Creates or reuses an Ubuntu virtual environment.
4. Installs the Linux dependencies.
5. Runs all tests.
6. Runs the paired heterogeneous-capacity burst experiment.
7. Creates delivery, delay and routing-update graphs.
8. Displays the graphs and exact averages directly in the notebook.

Nothing outside this notebook is required after Ubuntu/WSL is installed.

In [ ]:
from pathlib import Path
import base64, json, os, subprocess, shlex, zipfile, shutil
from IPython.display import display, Image, Markdown, HTML

ROOT=Path.cwd().resolve()
RUN_ROOT=ROOT/'taro_one_notebook_run'
RUN_ROOT.mkdir(exist_ok=True)
def wsl_path(p):
    win=str(Path(p).resolve()).replace('\\','/')
    r=subprocess.run(['wsl.exe','-d','Ubuntu','--','wslpath','-a','-u',win],text=True,capture_output=True)
    if r.returncode: raise RuntimeError(r.stderr)
    return r.stdout.strip()
WSL_RUN=wsl_path(RUN_ROOT)
print('Windows folder:',ROOT)
print('Ubuntu folder:',WSL_RUN)


## 1. Extract the embedded implementation

The notebook carries the taro package and tests inside a compressed bundle. Existing extracted files are reused; missing files are extracted.

In [ ]:
SOURCE_BUNDLE_B64='...'
bundle=RUN_ROOT/'source_bundle.zip'
if not (RUN_ROOT/'taro'/'targeted_experiment.py').exists() or not (RUN_ROOT/'requirements-linux.txt').exists():
    bundle.write_bytes(base64.b64decode(SOURCE_BUNDLE_B64))
    tmp_extract=RUN_ROOT/'_bundle_extract'
    with zipfile.ZipFile(bundle) as z: z.extractall(tmp_extract)
    source_root=tmp_extract/'CNET'
    if source_root.exists():
        for item in source_root.iterdir():
            target=RUN_ROOT/item.name
            if item.is_dir(): shutil.copytree(item,target,dirs_exist_ok=True)
            else: shutil.copy2(item,target)
    for item in tmp_extract.iterdir():
        if item.name == 'CNET':
            continue
        target=RUN_ROOT/item.name
        if item.is_dir(): shutil.copytree(item,target,dirs_exist_ok=True)
        else: shutil.copy2(item,target)
    shutil.rmtree(tmp_extract)
    bundle.unlink()
else:
    print('Using existing embedded source extraction.')
print('Source ready:',RUN_ROOT/'taro')


## 2. Create or reuse Ubuntu dependencies

This cell checks the Ubuntu virtual environment. If it is missing, it creates it and installs the CPU dependency list. If it exists, it is reused.

In [ ]:
requirements_wsl = wsl_path(RUN_ROOT / 'requirements-linux.txt')
home = subprocess.check_output(['wsl.exe', '-d', 'Ubuntu', '--', 'bash', '-lc', 'printf %s "$HOME"'], text=True).strip()
probe = 'import numpy, networkx, pandas, scipy, matplotlib, os_ken, torch; assert torch.__version__.split("+")[0] == "2.15.0"'
candidates = [f'{home}/taro-venv/bin/python', f'{WSL_RUN}/venv/bin/python']
WSL_PY = None
for candidate in candidates:
    check = subprocess.run(['wsl.exe', '-d', 'Ubuntu', '--', candidate, '-c', probe], capture_output=True)
    if check.returncode == 0:
        WSL_PY = candidate
        print('Using existing Ubuntu environment:', WSL_PY, flush=True)
        break
if WSL_PY is None:
    WSL_PY = candidates[-1]
    setup = f'cd {shlex.quote(WSL_RUN)} && python3 -m venv venv && ./venv/bin/python -m pip install -r {shlex.quote(requirements_wsl)}'
    print('Installing required packages in Ubuntu. This may take several minutes.', flush=True)
    subprocess.run(['wsl.exe', '-d', 'Ubuntu', '--', 'bash', '-lc', setup], check=True)
    subprocess.run(['wsl.exe', '-d', 'Ubuntu', '--', WSL_PY, '-c', probe], check=True)
print('Ubuntu environment is ready.')


## 3. Run the complete verification suite

These tests cover traffic conservation, capacity limits, failed links, PPO updates, checkpoint loading, destination-specific queues and targeted-experiment conservation rules.

In [ ]:
test_cmd=f"cd {shlex.quote(WSL_RUN)} && {shlex.quote(WSL_PY)} -m unittest discover -s tests -v"
result=subprocess.run(['wsl.exe','-d','Ubuntu','--','bash','-lc',test_cmd],text=True,capture_output=True)
print(result.stdout)
if result.returncode: print(result.stderr); raise RuntimeError('Tests failed')


## 4. Run the paired targeted scenario

The graph was selected during an exploratory search. The update threshold was selected on separate workload seeds 31 to 33. Workload seeds 101 to 110 vary the workload on the same graph. Each method receives identical traffic for each seed; all metrics are saved.

In [ ]:
results_file=RUN_ROOT/'artifacts'/'targeted_burst'/'results.json'
if results_file.exists():
    print('Using existing targeted experiment results.')
else:
    run_cmd=f"cd {shlex.quote(WSL_RUN)} && OMP_NUM_THREADS=1 {shlex.quote(WSL_PY)} -m taro.targeted_experiment"
    result=subprocess.run(['wsl.exe','-d','Ubuntu','--','bash','-lc',run_cmd],text=True,capture_output=True)
    print(result.stdout)
    if result.returncode: print(result.stderr); raise RuntimeError('Targeted experiment failed')
payload=json.loads(results_file.read_text())
print(payload['note'])
print('Seeds:',payload['workload_seeds'])


## 5. Delivery graph

Higher is better. This shows the fraction of offered traffic delivered after the active period and drain period. Each method uses the same ten workload seeds.

In [ ]:
out=RUN_ROOT/'artifacts'/'targeted_burst'
display(Image(filename=str(out/'delivery.png')))


## 6. Delay graph

Lower is better. This is simulated delay for delivered traffic, not packet-level controller measurement. Read it together with delivery fraction.

In [ ]:
display(Image(filename=str(out/'delay.png')))


## 7. Routing-update graph

Lower is better. This is a simulator routing-entry-change proxy, not a direct OpenFlow message count. It measures the stability benefit of the thresholded TARO policy.

In [ ]:
display(Image(filename=str(out/'updates.png')))


## 8. Exact statistics

This table is generated from the saved run records. It shows where TARO is better and where it pays a cost.

In [ ]:
means=json.loads((out/'means.json').read_text())
rows=['| Method | Delivery fraction | Mean delay | Changed entries |','|---|---:|---:|---:|']
for method in ('taro-stable','taro-heuristic','ecmp','ospf-style','load-aware'):
    rows.append(f"| {method} | {means['delivery_fraction'][method]:.4f} | {means['mean_delay'][method]:.4f} | {means['changed_entries'][method]:.1f} |")
display(Markdown('\n'.join(rows)))
taro=means['delivery_fraction']['taro-stable']
ospf=means['delivery_fraction']['ospf-style']
original=means['changed_entries']['taro-heuristic']
stable=means['changed_entries']['taro-stable']
display(Markdown(f"Measured targeted result: Stable TARO delivers {(taro-ospf)*100:.3f} percentage points more traffic than OSPF-style routing in this scenario and reduces changed entries by {(original-stable)/original*100:.1f}% versus the original TARO heuristic."))


## 9. Research-paper comparison

The supplied Phase 2 report identifies Grace, GTSR, ERA-MADDPG, CLPE-TE, GMHA-DRL, PathGNN, LARRI, GRL-RR and TELGEN.

The specific opportunity tested here is the joint quality and control-plane-stability question. The Phase 2 review notes that many papers focus on delay, loss, topology transfer or convergence, while controller update cost is less often measured. The local targeted experiment therefore evaluates delivery and route changes together.

This notebook does not claim direct numerical superiority over those papers because their original implementations were not run under the same topology, traffic, training budget and metrics. The comparison is a local reproducible experiment against implemented baselines.

## 10. Reproducibility files

All generated outputs are under taro_one_notebook_run:

- artifacts/targeted_burst/results.json
- artifacts/targeted_burst/means.json
- artifacts/targeted_burst/delivery.png
- artifacts/targeted_burst/delay.png
- artifacts/targeted_burst/updates.png

The source and tests are embedded in this notebook, so it can recreate the run folder from scratch.